> **INSTRUCTOR SOLUTIONS** — do not share with learners before the session.

# Part 7 · Notebook 01 — The strategy framework and the first-look evaluator

**Sessions:** S1 (Strategy framework, spec template & first-look evaluator) · [Lesson plan](../../docs/lessons/PART_07_STRATEGY_LIBRARY.md) · graded labs in [`labs/part07/`](../../labs/part07/)

**You will:**
1. Write a strategy as a registered class that only emits intents.
2. Write the first-look evaluator with the next-bar fill rule.
3. See what a same-bar fill does to a worthless signal.
4. Write the mandatory next-bar execution test and catch a strategy that peeks.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic, built from regimes you know, and every strategy here is a **hypothesis** with a first-look evaluation: the honest backtest comes in Part 8.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p7lib.py is in notebooks/part07/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p7lib as p

p.use_course_style()

## 1. A market with known regimes

Twenty-four half-year blocks, each either **trending** (a steady drift up or down) or **range-bound** (pulled back to where it started), and either **calm** (10% vol) or **volatile** (28% vol). The labels are in the data, so later we can ask which strategy works where.

In [ ]:
bars = p.regime_market()
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(bars.index, bars.close, lw=0.8, color="black")
for i in range(0, len(bars), 125):
    blk = bars.iloc[i:i + 125]
    ax.axvspan(blk.index[0], blk.index[-1], alpha=0.12, color=p.PALETTE[2] if blk.trend.iloc[0] else p.PALETTE[1], lw=0)
ax.set_title("Green: trending blocks · orange: range-bound blocks"); plt.show()
bars.head(3)

## 2. One class, three runtimes

A strategy never calls a broker. It sees the bars **up to now** through `self.ctx.history(n)`, and says what it wants with `self.target(weight, reason)`: an *intent*. The research runner, the backtester (Part 8) and the live OMS (Part 4) all drive the same class. `@p.register("name")` adds it to the registry, and `params` declares defaults with their allowed ranges (`name: (default, min, max)`) so configs can be validated and optimizers know the search space.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

Write `on_bar`: once there are at least `slow` bars of history, target weight **1** when the fast SMA of the closes is above the slow SMA, else **0** (use `p.sma(closes, n)[-1]`). Emit nothing during the warm-up.

In [ ]:
p.REGISTRY.pop("sma_cross", None)                # so the cell can be re-run

@p.register("sma_cross")
class SmaCross(p.Strategy):
    params = {"fast": (20, 5, 100), "slow": (100, 20, 300)}

    def on_bar(self):
        closes = self.ctx.history(self.p["slow"])["close"].to_numpy()
        if len(closes) < self.p["slow"]:
            return
        up = p.sma(closes, self.p["fast"])[-1] > p.sma(closes, self.p["slow"])[-1]
        self.target(1.0 if up else 0.0, "fast above slow" if up else "fast below slow")

decided = p.attempt(p.run, SmaCross, bars)
fast, slow = p.sma(bars.close, 20), p.sma(bars.close, 100)
ref_dec = np.where(np.nan_to_num(fast) > np.nan_to_num(slow, nan=np.inf), 1.0, 0.0)
mine = p.check("sma_cross decisions", decided, ref_dec)
print(f"in the market {mine.mean():.0%} of bars")

In [ ]:
try:
    p.run(p.REGISTRY["sma_cross"], bars, fast=500)
except ValueError as e:
    print("config validation:", e)

## 3. The first-look evaluator

A decision made at bar `t`'s **close** can be filled at bar `t+1`'s **open** at the earliest. So the position held from open `t` to open `t+1` is `decided[t−1]` (flat on the first bar), and each change of position pays `cost_bps` on the turnover. Return the daily P&L array `pos · ret − turnover · cost_bps / 10,000`.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def first_look_pnl(decided, open_, cost_bps=2.0):
    decided = np.nan_to_num(np.asarray(decided, dtype=float))
    pos = np.zeros_like(decided)
    pos[1:] = decided[:-1]
    ret = np.zeros_like(open_)
    ret[:-1] = open_[1:] / open_[:-1] - 1         # open-to-open
    turnover = np.abs(np.diff(pos, prepend=0.0))
    return pos * ret - turnover * cost_bps / 1e4

o = bars.open.to_numpy()
mine = p.attempt(first_look_pnl, ref_dec, o)
mine = p.check("first-look P&L", mine, p.quick_eval(ref_dec, o)["pnl"])
p.summary(p.quick_eval(ref_dec, o))

## 4. Same-bar fills are fantasy

A worthless signal: "today closed higher than yesterday, so be long". Evaluate it honestly (fill at the next open), and then with the most common backtest bug, filling at the open **of the bar that produced the signal**: the position is taken before the close that decided it.

In [ ]:
c = bars.close.to_numpy()
today_up = np.r_[False, c[1:] > c[:-1]].astype(float)
honest = p.quick_eval(today_up, o)
peeking = p.quick_eval(np.r_[today_up[1:], 0.0], o)     # shifting the decision one bar earlier = filling on its own bar
print(f"next-bar fill: Sharpe {honest['sharpe']:+.2f}")
print(f"same-bar fill: Sharpe {peeking['sharpe']:+.2f}  ← a result like this is a bug until proven otherwise")

## 5. The mandatory next-bar execution test

The runner only hands a strategy `history()`, but nothing stops a careless strategy from reading `self.ctx.bars` directly. The test that catches any such leak: run the strategy on the data **cut at bar t**, and check that its decisions up to `t` are identical to the ones it made on the full data. A causal strategy can't tell the difference; a peeking one can.

In [ ]:
p.REGISTRY.pop("peeker", None)

@p.register("peeker")
class Peeker(p.Strategy):
    """Looks one bar ahead through ctx.bars: it would never pass review."""
    def on_bar(self):
        b, t = self.ctx.bars, self.ctx.t
        if t + 1 < len(b):
            self.target(1.0 if b.close.iloc[t + 1] > b.close.iloc[t] else 0.0, "knows tomorrow")

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def next_bar_test(strategy_cls, bars, cuts=(500, 1200, 2000)):
    full = p.run(strategy_cls, bars)
    for t in cuts:
        cut = p.run(strategy_cls, bars.iloc[: t + 1])
        if not np.array_equal(cut, full[: t + 1]):
            return False
    return True

mine = [p.attempt(next_bar_test, p.REGISTRY["sma_cross"], bars), p.attempt(next_bar_test, p.REGISTRY["peeker"], bars)]
mine = p.check("next_bar_test", mine, [True, False])
print(f"sma_cross passes: {mine[0]};  peeker passes: {mine[1]}  (its first-look Sharpe: {p.quick_eval(p.run(Peeker, bars), o)['sharpe']:.1f})")

## Wrap-up

* Strategies emit intents; the risk engine and OMS decide what is sent.
* Declared parameter ranges make configs checkable and optimization honest.
* Next-bar fills in the evaluator, and a truncation test on every strategy (it is a pass/fail criterion of milestone M3b).
* The first-look evaluator is deliberately crude; Part 8 replaces it.
* Graded version: `labs/part07/week23_framework_momentum` (base class, registry, YAML configs, runner, `quick_eval`, the next-bar test).